### **Semana 11: arquitectura multiagente, contratos y evaluación**

**Pregunta de la sesión:** ¿qué aporta dividir una tarea entre agentes si la recuperación inicial es la misma, y qué nuevas fallas pueden aparecer?

Este cuaderno diferencia la recuperación de HotpotQA, las pruebas de contratos y la inferencia real con Qwen. No presenta la orquestación determinista como un agente LLM.

#### **1. Preparar el entorno**

Los experimentos usan los archivos originales de HotpotQA preparados por el docente con `python -m Semana11.multiagente_real --preparar`. La carga debe fallar si falta el archivo o no coincide su hash. No se fabrican ejemplos para ejecutar el cuaderno.

In [1]:
from pathlib import Path
import json, os, sys
root = Path.cwd().resolve()
for candidate in (root, *root.parents):
    if (candidate / 'Semana11/multiagente_real.py').is_file():
        root = candidate
        break
else:
    raise FileNotFoundError('Inicia JupyterLab desde el repositorio CC-0F4')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from Semana11.multiagente_real import (check_dataset, ranked_evidence, retrieval_rows,
    summarize_retrieval, support_recall, gold_facts, evaluate_retrieval,
    validate_answer, select_pilot, pilot, MODEL_ID)
from Semana11.contratos import Blackboard, ContractError, make_message, validate_selection
examples = check_dataset()
print('Directorio:', root)
print('Preguntas originales verificadas:', len(examples))

Directorio: /workspace/CC-0F4
Preguntas originales verificadas: 7405


#### **2. HotpotQA: pregunta, contexto y evidencia original**

HotpotQA registra pregunta, respuesta humana, párrafos distractores y hechos de apoyo. Cada hecho de apoyo se identifica por el título del documento y el índice de la oración. La variante distractor no representa recuperación abierta sobre Wikipedia.

El recuperador utilizará solamente `question` y `context`. No debe acceder a `answer` ni a `supporting_facts` antes de la evaluación.

In [2]:
# Inspección de un registro humano real, sin usar la respuesta como entrada al recuperador.
example = examples[0]
print('ID:', example['_id'])
print('Tipo:', example['type'], '| dificultad:', example['level'])
print('Pregunta:', example['question'])
print('Número de párrafos:', len(example['context']))
print('Número de hechos gold:', len(example['supporting_facts']))

ID: 5a8b57f25542995d1e6f1371
Tipo: comparison | dificultad: hard
Pregunta: Were Scott Derrickson and Ed Wood of the same nationality?
Número de párrafos: 10
Número de hechos gold: 2


#### **3. Agente único y supervisor con trabajadores**

Agente único: pregunta y evidencia disponible, una generación de respuesta y citas.

Supervisor: primera generación para seleccionar identificadores, validación y registro, segunda generación para responder con fragmentos seleccionados.

El supervisor actual aplica reglas en Python. Los trabajadores solo se consideran dirigidos por LLM cuando se ejecuta el modo Qwen. El controlador no es por sí solo un modelo autónomo.

#### **4. Acciones y mensajes observables**

Una transición de coordinación puede modelarse como una función de estado, entrada y mensaje:

$$s_{t+1} = F(s_t, m_t, o_t).$$

El mensaje aceptado debe indicar `question_id`, `role`, `selected_ids` y huellas de procedencia. La pizarra permite reconstruir el orden de aceptación. Un identificador no observado debe ser rechazado, aunque aparezca en una respuesta JSON correctamente formada.

In [3]:
observed = ranked_evidence(example, top_k=14)
print('Oraciones disponibles:', len(observed))
for row in observed[:4]:
    print(row['id'], row['title'], row['sent_id'], row['text'][:140])
assert all(isinstance(row['id'], str) for row in observed)

Oraciones disponibles: 14
E00 Woodson, Arkansas 4  Woodson is adjacent to the Wood Plantation, the largest of the plantations own by Ed Wood Sr.
E01 Woodson, Arkansas 3  Woodson and its accompanying Woodson Lake and Wood Hollow are the namesake for Ed Wood Sr., a prominent plantation owner, trader, and busin
E02 Ed Wood (film) 0 Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmm
E03 Conrad Brooks 3  He took a break from acting during the 1960s and 1970s but due to the ongoing interest in the films of Ed Wood, he reemerged in the 1980s a


#### **5. Contrato de traspaso y pizarra**

La pizarra guarda mensajes aceptados y no permite cambiar los datos validados a posteriori. Se añade una huella SHA-256 del texto recuperado y su procedencia. Esta verificación de integridad no acredita relevancia semántica.

In [4]:
board = Blackboard(example['_id'])
selected = [row['id'] for row in observed[:2]]
msg = make_message(example['_id'], 'evidence_worker', selected, observed)
board.append(msg, observed)
print('Mensajes aceptados:', len(board.messages))
print('IDs registrados:', board.messages[0]['selected_ids'])
assert board.messages[0]['selected_ids'] == selected

Mensajes aceptados: 1
IDs registrados: ['E00', 'E01']


#### **6. Validación: el supervisor debe poder rechazar**

Una regla de seguridad debe estar en código, no depender solo de que el modelo siga instrucciones. Compara una cita observada y una no observada. La siguiente celda inyecta errores para probar contratos, no añade preguntas sintéticas al benchmark.

In [5]:
# Inyección de falla de contrato, separada de cualquier resultado experimental.
try:
    validate_selection(['ID_NO_OBSERVADO'], observed)
except ContractError as exc:
    print('Rechazo esperado:', exc)
else:
    raise AssertionError('El contrato aceptó una cita inexistente')
valid, reason, answer, cited = validate_answer(
    {'answer':'Revisar evidencia', 'cited_ids': [observed[0]['id']]}, observed)
print('Finalización estructural:', valid, reason)
assert valid  # No demuestra que la respuesta de ejemplo sea factual.

Rechazo esperado: CITA_NO_OBSERVADA
Finalización estructural: True FINALIZACION_ESTRUCTURAL


#### **7. Recuperación BM25 sin filtrar las anotaciones**

El algoritmo recorre las oraciones de los párrafos suministrados con una pregunta. Calcula frecuencia de términos, frecuencia documental y una puntuación BM25. El gold queda fuera de la función de recuperación.

La unidad de evaluación es la oración, no el pasaje completo. Dos oraciones del mismo documento son dos unidades distintas.

In [6]:
ranked = ranked_evidence(example, 14)
assert ranked == ranked_evidence(example, 14)
print('Ranking determinista para:', example['_id'])
for x in ranked[:3]:
    print(x['id'], f"BM25={x['bm25']:.3f}", x['text'][:120])

Ranking determinista para: 5a8b57f25542995d1e6f1371
E00 BM25=6.339  Woodson is adjacent to the Wood Plantation, the largest of the plantations own by Ed Wood Sr.
E01 BM25=5.898  Woodson and its accompanying Woodson Lake and Wood Hollow are the namesake for Ed Wood Sr., a prominent plantation owne
E02 BM25=5.127 Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnn


#### **8. Métricas para evidencia de varios saltos**

Con G como conjunto de hechos de apoyo humanos y R como oraciones recuperadas:

$$\mathrm{Recall} = |G \cap R|/|G|.$$

AnyHit vale uno si al menos una oración gold está en R. AllHit vale uno solo si todas las oraciones gold están presentes. No equipares AnyHit con capacidad de contestar la pregunta.

In [7]:
# La anotación original se abre únicamente después del ranking.
print('Respuesta humana:', example['answer'])
print('Hechos humanos:', sorted(gold_facts(example)))
print('Recall de soporte:', round(support_recall(example, ranked), 4))
print('Observado:', [(x['title'],x['sent_id']) for x in ranked[:4]])

Respuesta humana: yes
Hechos humanos: [('Ed Wood', 0), ('Scott Derrickson', 0)]
Recall de soporte: 0.5
Observado: [('Woodson, Arkansas', 4), ('Woodson, Arkansas', 3), ('Ed Wood (film)', 0), ('Conrad Brooks', 3)]


#### **9. Evaluación completa, sin simulación**

Se reutilizan el resumen y las trazas reales si ambos existen. Si no existe ninguno, se ejecuta por primera vez el benchmark de 7.405 preguntas. No se sustituyen datos ni se generan resultados artificiales. Si solo existe uno de los archivos, se debe recuperar la evidencia faltante antes de continuar.

In [ ]:
results_dir = root / 'Semana11' / 'resultados'
summary_path = results_dir / 'hotpot_recuperacion_resumen.json'
rows_path = results_dir / 'hotpot_recuperacion_7405_por_consulta.jsonl'
if summary_path.is_file() and rows_path.is_file():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('Resultados de recuperación almacenados. No se repite el benchmark.')
elif not summary_path.exists() and not rows_path.exists():
    summary = evaluate_retrieval()
    print('Primera evaluación de recuperación.')
else:
    raise FileNotFoundError('Evidencia incompleta: restaura ambos archivos de recuperación antes de continuar.')
assert summary['questions'] == 7405
assert summary['top_k'] == 14
assert summary['source_sha256'] == '4e9ecb5c8d3b719f624d66b60f8d56bf227f03914f5f0753d6fa1b359d7104ea'
print('Preguntas:', summary['questions'])
for metric, value in summary['retrieval'].items():
    print(metric, value)


#### **10. Desglose por tipo y dificultad**

Los promedios globales pueden ocultar comportamientos distintos entre preguntas de tipo `bridge` y `comparison`, o entre niveles de dificultad. El desglose proviene de los metadatos originales, no de etiquetas construidas después de observar predicciones.

In [9]:
for name, group in summary['by_type'].items():
    print('Tipo:', name, '| preguntas:', group['questions'],
          '| AllHit:', round(group['all_support_rate'], 4))
for name, group in summary['by_level'].items():
    print('Nivel:', name, '| preguntas:', group['questions'],
          '| Recall:', round(group['mean_support_recall'], 4))

Tipo: bridge | preguntas: 5918 | AllHit: 0.5874
Tipo: comparison | preguntas: 1487 | AllHit: 0.7438
Nivel: hard | preguntas: 7405 | Recall: 0.819


#### **11. Limites de interpretación y costo**

El modelo local genera tokens de entrada y salida y consume tiempo de inferencia. El supervisor con trabajadores necesita normalmente dos llamadas, mientras que el agente único necesita una. La comparación no iguala automáticamente presupuesto ni complejidad.

Ni EM/F1 sobre respuestas ni las coincidencias de citas bastan para establecer la veracidad de todas las afirmaciones. Se requiere evaluación de respaldo factual independiente para una conclusión de grounding.

#### **12. Piloto opcional, nunca simulado**

Por defecto se consultan los resultados reales del piloto de 16 preguntas y sus trazas, si están disponibles. No se descarga Qwen ni se ejecutan inferencias al consultar esos archivos. Solo se realiza una nueva ejecución si se activa explícitamente `CC0F4_RUN_REAL_LLM=1`. 

Las variables `CC0F4_S11_LLM_N` y `CC0F4_S11_MODEL` configuran una nueva ejecución; no cambian los resultados almacenados. Una nueva ejecución puede sobrescribir sus archivos de salida, por lo que se debe respaldar primero la evidencia previa. Las finalizaciones inválidas se registran sin repararlas.

In [ ]:
results_dir = root / 'Semana11' / 'resultados'
pilot_path = results_dir / 'hotpot_piloto_16_resumen.json'
traces_path = results_dir / 'hotpot_piloto_16_trazas.jsonl'
if os.getenv('CC0F4_RUN_REAL_LLM', '0') == '1':
    count = int(os.getenv('CC0F4_S11_LLM_N', '16'))
    report = pilot(count, os.getenv('CC0F4_S11_MODEL', MODEL_ID))
    print('Nueva ejecución experimental solicitada explícitamente.')
    print(json.dumps(report, ensure_ascii=False, indent=2))
elif pilot_path.is_file() and traces_path.is_file():
    report = json.loads(pilot_path.read_text(encoding='utf-8'))
    assert report['questions'] == 16
    assert report['dataset_sha256'] == '4e9ecb5c8d3b719f624d66b60f8d56bf227f03914f5f0753d6fa1b359d7104ea'
    assert report['status'] == 'PILOTO_REAL_EXPLORATORIO'
    print('Piloto real almacenado. No se realizan nuevas inferencias.')
    print(json.dumps(report, ensure_ascii=False, indent=2))
    print('Trazas:', traces_path)
elif pilot_path.exists() or traces_path.exists():
    raise FileNotFoundError('Evidencia del piloto incompleta: restaura el resumen y las trazas.')
else:
    report = None
    print('No hay resultados del piloto de 16 preguntas. No se realizaron inferencias en esta sesión.')


#### **13. Cierre de la teoría**

Explica por qué una cita estructuralmente válida puede ser equivocada, qué falla cuando se conserva una referencia mutable en la pizarra y por qué AllHit es una medida más estricta que AnyHit.

Diferencia claramente los tres resultados de aprendizaje: recuperación de evidencia, cumplimiento de contratos y respuestas del modelo. Ninguna prueba unitaria debe confundirse con una evaluación científica de agentes.